This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [2]:
import great_expectations as gx
context = gx.get_context()
import logging

AttributeError: module 'great_expectations' has no attribute 'get_context'

In [3]:
import yaml

In [4]:
from datetime import date

In [5]:
logging.basicConfig(level=logging.DEBUG, force = True)

NameError: name 'logging' is not defined

In [6]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [6]:
with open("great_expectations/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [7]:
datasource_config.get("project")

'gfw-google-827'

In [8]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterYearAndMonthAndDay.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterYearAndMonthAndDay.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.

In [9]:
gx_datasource.get_asset_names()

{'messages',
 'satellite_timing_offsets',
 'segs_activity',
 'segs_activity_daily',
 'ssvids_identities',
 'ssvids_identities_daily',
 'stats_daily',
 'vessel_info'}

In [10]:
context.list_expectation_suite_names()

['gfw-google-827.segments-daily',
 'gfw-google-827.segments-yearly',
 'pipe_ais_v3_alpha_published.satellite_timing_offsets',
 'pipe_ais_v3_alpha_published.segs_activity_daily',
 'pipe_ais_v3_alpha_published.vessel_info']

In [26]:
segs_activity_daily_asset = gx_datasource.get_asset("segs_activity_daily")

In [1]:
segs_activity_daily_br = segs_activity_daily_asset.build_batch_request({'date': date.fromisoformat('2023-01-01')})

NameError: name 'segs_activity_daily_asset' is not defined

In [28]:
selected_expectation_suit_name = 'gfw-google-827.constraints.segs_activity_daily'
selected_expectation_suit = context.get_expectation_suite(selected_expectation_suit_name)

In [29]:
segs_activity_daily_cp = context.add_checkpoint(
    name=f"{selected_expectation_suit_name}-checkpoint",
    validations=[
     {
      "batch_request": segs_activity_daily_br
     }
    ],
    expectation_suite_name=selected_expectation_suit_name)

DEBUG:great_expectations.data_context.util:(instantiate_class_from_config) module_name -> great_expectations.checkpoint


In [30]:
segs_activity_daily_cp.run()

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:google.cloud.bigquery.opentelemetry_tracing:This service is instrumented using OpenTelemetry. OpenTelemetry or one of its components could not be imported; please add compatible versions of opentelemetry-api and opentelemetry-instrumentation packages in order to get BigQuery Tracing data.
DEBUG:urllib3.util.retry:Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)
DEBUG:google.auth.transport.requests:Making request: POST https://oauth2.googleapis.com/token
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): oauth2.googleapis.com:443
DEBUG:urllib3.connectionpool:https://oauth2.googleapis.com:443 "POST /token HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): bigquery.googleapis.com:4

InvalidBatchRequestError: Validator could not be created because BatchRequest returned an empty batch_list.
                Please check your parameters and try again.